In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import torch 
from datasets import load_dataset
import cv2
from tqdm.auto import tqdm  
from PIL import Image
import sys 
import io
from scipy.ndimage import gaussian_filter

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
## fetching dataset kricko/cleaned_auditor
## using model kricko/Auditor_Model to build heatmaps
## writing into ShreyashDhoot/KTO_training

In [ ]:
#settnig up the auditor 
!git clone https://huggingface.co/kricko/Auditor_Model

In [ ]:
dataset=load_dataset("kricko/cleaned_auditor",streaming=True)

In [ ]:

dataset_train=dataset["train"]

first_example=next(iter(dataset_train))

print(first_example)

plt.imshow(first_example["image"])

In [ ]:
sys.path.append('Auditor_Model')

In [ ]:
import json
import os

# 1. Path to the oversized vocab you just loaded
original_vocab_path = "/kaggle/working/Auditor_Model/vocab.json"

# 2. Load the large vocabulary
with open(original_vocab_path, "r") as f:
    large_vocab = json.load(f)

# 3. Filter the vocabulary to ONLY include words with an index under 15807.
# This guarantees the size will be exactly 15807, and prevents any "Index Out of Bounds" 
# errors when the words are passed into the embedding layer.
fixed_vocab = {word: idx for word, idx in large_vocab.items() if idx < 15807}

# 4. Save this new, perfectly sized vocabulary
NEW_VOCAB_PATH = "/kaggle/working/Auditor_Model/vocab_15807.json"
with open(NEW_VOCAB_PATH, "w") as f:
    json.dump(fixed_vocab, f)

print(f"Created new vocab file with {len(fixed_vocab)} words at {NEW_VOCAB_PATH}")

In [ ]:
#setting up auditor inference 
import auditor_inference 
from auditor_inference import audit_image
pil_img = first_example["image"]

# 2. Create a virtual file in memory
img_buffer = io.BytesIO()

# 3. Save the image to this memory buffer instead of the hard drive
pil_img.save(img_buffer, format="PNG")

# 4. Rewind the buffer's "cursor" to the beginning so it can be read
img_buffer.seek(0)
auditor_inference.np = np

results = audit_image(
    model_path="/kaggle/working/Auditor_Model/complete_auditor_best.pth",
    image_path=img_buffer,
    vocab_path=NEW_VOCAB_PATH,
    prompt=first_example["prompt"],
    return_heatmaps=True
)

print(results)
plt.imshow(results["adversarial_heatmap"])

# Heatmaps are available as numpy arrays (original image size)
# results["adversarial_heatmap"]
# results["category_heatmaps"]["Violence"]


In [ ]:
heatmap=results["adversarial_heatmap"]
dynamic_thresh = np.percentile(heatmap,60)

# Create mask based on that specific image's top pixels
binary_mask = (heatmap >= dynamic_thresh).astype(np.uint8)
kernel = np.ones((5,5), np.uint8)
dilated_mask = cv2.dilate(binary_mask, kernel, iterations=12)

# 3. Feather: Soften the edges so the Ginger twins don't have "cut-out" lines
feathered_mask = gaussian_filter(dilated_mask.astype(float), sigma=5.0)
plt.imshow(first_example["image"])
plt.imshow(feathered_mask,cmap='jet', alpha=0.5)

![Flowchart](/kaggle/working/pipeline_flowchart.png)

In [ ]:
import torch
import torch.nn.functional as F
from auditor_inference import SimpleTokenizer, CompleteMultiTaskAuditor, predict_single

# --- Load model ONCE ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Using device: {DEVICE}")

tokenizer = SimpleTokenizer(vocab_path=NEW_VOCAB_PATH)
model = CompleteMultiTaskAuditor(num_classes=5, vocab_size=len(tokenizer.word_to_idx))
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE).eval()
print("✅ Model loaded on GPU once.")

In [ ]:
import io
import cv2
import numpy as np
from datasets import load_dataset, Dataset, Features, Image, Value
from scipy.ndimage import gaussian_filter
from huggingface_hub import HfApi
import auditor_inference
from auditor_inference import audit_image

# --- Configuration ---
SOURCE_DATASET = "kricko/cleaned_auditor"
TARGET_DATASET = "ShreyashDhoot/KTO_trial"
MODEL_PATH = "/kaggle/working/Auditor_Model/complete_auditor_best.pth"
BATCH_SIZE = 2500

auditor_inference.np = np

api = HfApi()

# --- Count existing parquet shards to resume batch numbering ---
try:
    existing_files = api.list_repo_files(TARGET_DATASET, repo_type="dataset")
    existing_parquets = [f for f in existing_files if f.endswith(".parquet")]
    batch_counter = len(existing_parquets)
except Exception:
    batch_counter = 0

# --- 1. Fetch existing IDs to avoid duplicates (Resumable Logic) ---
print("🔍 Checking existing progress on Hugging Face...")
try:
    existing_ds = load_dataset(TARGET_DATASET, split="train", columns=["id"])
    processed_ids = set(str(i) for i in existing_ds["id"])
    print(f"✅ Found {len(processed_ids)} already processed examples. Resuming...")
except Exception as e:
    processed_ids = set()
    print(f"🆕 No existing dataset found or error loading (Starting fresh).")

# --- 2. Processing Function (Your specific Auditor logic) ---
def process_example(example):
    pil_img = example["image"]
    prompt = example["prompt"]

    # Audit logic
    img_buffer = io.BytesIO()
    pil_img.save(img_buffer, format="PNG")
    img_buffer.seek(0)

    results = predict_single(model,tokenizer,img_buffer,prompt=prompt,return_heatmaps=True)

    # Create Mask
    heatmap = results["adversarial_heatmap"]
    dynamic_thresh = np.percentile(heatmap, 75)
    binary_mask = (heatmap >= dynamic_thresh).astype(np.uint8) * 255

    # Morphological operations
    kernel = np.ones((5,5), np.uint8)
    dilated = cv2.dilate(binary_mask, kernel, iterations=2)

    # Feathering
    feathered_mask_np = gaussian_filter(dilated.astype(float), sigma=5.0).astype(np.uint8)

    # Image Removal Logic (Black out the feathered area)
    cv_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    removal_mask = (feathered_mask_np > 50) 
    img_removed = cv_img.copy()
    img_removed[removal_mask] = 0 
    img_removed_pil = cv2.cvtColor(img_removed, cv2.COLOR_BGR2RGB)

    return {
        "image": pil_img,
        "prompt": prompt,
        "id": str(example["id"]),
        "disturbing": int(example["disturbing"]),
        "hate": int(example["hate"]),
        "illegal activity": int(example["illegal activity"]),
        "safe": int(example["safe"]),
        "sexual": int(example["sexual"]),
        "violence": int(example["violence"]),
        "feathered_mask": feathered_mask_np,
        "image_masked_removed": img_removed_pil
    }

# --- 3. Define Schema ---
features = Features({
    "image": Image(),
    "prompt": Value("string"),
    "id": Value("string"),
    "disturbing": Value("int8"),
    "hate": Value("int8"),
    "illegal activity": Value("int8"),
    "safe": Value("int8"),
    "sexual": Value("int8"),
    "violence": Value("int8"),
    "feathered_mask": Image(),
    "image_masked_removed": Image(),
})

# --- Helper: upload a Dataset batch as a new parquet shard ---
def upload_shard(ds, shard_index):
    buffer = io.BytesIO()
    ds.to_parquet(buffer)
    buffer.seek(0)
    api.upload_file(
        path_or_fileobj=buffer,
        path_in_repo=f"data/train-{shard_index:05d}-of-NNNNN.parquet",
        repo_id=TARGET_DATASET,
        repo_type="dataset",
    )
    print(f"💎 Shard {shard_index + 1} uploaded. Memory cleared.")

# --- 4. Execution Logic ---
print("🚀 Starting Stream...")
source_ds = load_dataset(SOURCE_DATASET, streaming=True, split="train")

collected_data = []
total_newly_processed = 0

for raw_example in source_ds:
    current_id = str(raw_example["id"])

    # SKIP logic
    if current_id in processed_ids:
        continue

    try:
        processed = process_example(raw_example)
        collected_data.append(processed)
        total_newly_processed += 1

        if total_newly_processed % 50 == 0:
            print(f"✅ Processed {total_newly_processed} new images...")

        # Batch upload trigger
        if len(collected_data) >= BATCH_SIZE:
            print(f"📦 Batch reached {BATCH_SIZE}. Syncing to Hugging Face...")
            batch_ds = Dataset.from_list(collected_data, features=features)
            upload_shard(batch_ds, batch_counter)
            batch_counter += 1
            collected_data = []  # CRITICAL: Clear RAM

    except Exception as e:
        print(f"❌ Error at ID {current_id}: {e}")

# --- 5. Final Cleanup ---
if collected_data:
    print(f"📦 Uploading final partial batch of {len(collected_data)}...")
    final_batch = Dataset.from_list(collected_data, features=features)
    upload_shard(final_batch, batch_counter)
    batch_counter += 1

print(f"✨ Task Complete! Total newly added: {total_newly_processed}")

In [ ]:
from huggingface_hub import DatasetCard, DatasetCardData

# --- 6. Push Dataset Card (README.md) ---
card_data = DatasetCardData(
    language="en",
    license="mit",
    task_categories=["image-classification"],
    tags=["image", "safety", "adversarial", "inpainting", "kto"],
)

card_content = f"""---
{card_data.to_yaml()}
---

# KTO Training Dataset

Processed from [{SOURCE_DATASET}](https://huggingface.co/datasets/{SOURCE_DATASET}) using the Auditor model.

## Description
Each example contains the original image alongside adversarial heatmaps, feathered masks, 
and masked images with detected unsafe regions blacked out.

## Features
| Column | Type | Description |
|---|---|---|
| `image` | Image | Original input image |
| `prompt` | string | Text prompt associated with the image |
| `id` | string | Unique identifier |
| `disturbing` | int8 | Disturbing content score |
| `hate` | int8 | Hate content score |
| `illegal activity` | int8 | Illegal activity score |
| `safe` | int8 | Safe content score |
| `sexual` | int8 | Sexual content score |
| `violence` | int8 | Violence content score |
| `feathered_mask` | Image | Feathered adversarial mask (sigma=5, 75th percentile threshold) |
| `image_masked_removed` | Image | Original image with adversarial regions blacked out |

## Processing Details
- **Threshold**: 75th percentile of adversarial heatmap
- **Morphological dilation**: 5×5 kernel, 2 iterations
- **Feathering**: Gaussian blur sigma=5.0
- **Total examples**: {total_newly_processed}
"""

card = DatasetCard(card_content)
card.push_to_hub(TARGET_DATASET)
print("📋 Dataset card pushed successfully!")